# Repaso - Ingeniería Financiera (IIND-4414)
Notebook de funciones y conceptos para el Parcial 1.

**Temas:**
1. Extracción de datos (FRED, Yahoo Finance)
2. Log-retornos y volatilidad histórica
3. Pruebas estadísticas (Jarque-Bera, Ljung-Box, Engle)
4. GARCH(1,1)
5. Futuros y forwards (6 escenarios)
6. Black-Scholes-Merton (4 variantes)
7. Árboles binomiales (europeas y americanas)
8. Estrategias con opciones


In [ ]:
import numpy as np
import pandas as pd
import math

# Datos
import yfinance as yfin
from fredapi import Fred

# Gráficos
import matplotlib.pyplot as plt
from matplotlib import font_manager
plt.rcParams['font.family'] = 'Arial'

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import statsmodels.api as sm

# Estadística
from scipy.stats import norm
from scipy import stats

# GARCH
from arch import arch_model

# Engle ARCH-LM
from statsmodels.stats.diagnostic import het_arch

import warnings
warnings.filterwarnings('ignore')


In [ ]:
# API de FRED (Federal Reserve Economic Data)
fred = Fred(api_key='a3d8a45391e92cc5c30c9c255819c138')

## 1. Extracción de datos

Dos fuentes principales:
- **FRED** (fred.get_series): datos macro, commodities (ULSD, Brent, SOFR, etc.)
- **Yahoo Finance** (yfin.download): acciones, índices, tasas de cambio (COP=X)


In [ ]:
# ── Descargar serie desde FRED ──
# Argumentos: código de la serie, fecha inicio, fecha fin (opcional)

inicio = '2010-01-01'
fin = '2026-09-23'

df_fred = pd.DataFrame(fred.get_series(
    'DCOILBRENTEU',      # código FRED (ej: Brent, ULSD = DDFUELUSGULF)
    inicio,              # fecha inicio
    fin                  # fecha fin (opcional)
))
df_fred = df_fred.dropna()
df_fred.index.names = ['Date']
df_fred.columns = ['Close']

df_fred.tail()


In [ ]:
# ── Descargar serie desde Yahoo Finance ──
# Argumentos: ticker, start, end (opcional)

df_yf = yfin.download(
    'COP=X',             # ticker (ej: AAPL, ^GSPC, COP=X)
    start=inicio,        # fecha inicio
    end=fin,             # fecha fin (opcional)
    multi_level_index=False
)
df_yf.index.names = ['Date']

df_yf.tail()


### Close vs Adj Close

- **Close**: precio de cierre del día, sin ajustes.
- **Adj Close** (Adjusted Close): precio ajustado por dividendos y splits. Para acciones que pagan dividendos, usar Adj Close da retornos más precisos.
- **FRED** solo trae un precio (Close). Yahoo Finance trae ambos.
- **Regla**: para acciones, usar `Adj Close`. Para commodities, FX o datos FRED, usar `Close`.


## 2. Log-retornos y volatilidad histórica

El **log-retorno** mide el cambio porcentual logarítmico del precio:

$$r_t = \ln(P_t) - \ln(P_{t-1})$$

Son aditivos en el tiempo (a diferencia de retornos simples) y se usan en todos los modelos financieros.


In [ ]:
# ── Calcular log-retornos ──
# Usar 'Adj Close' para acciones, 'Close' para commodities/FX

def log_retornos(df, columna='Close'):
    """
    Calcula los log-retornos de una serie de precios.
    
    Parámetros:
        df       : DataFrame con la serie de precios
        columna  : nombre de la columna de precios ('Close' o 'Adj Close')
    Retorna:
        Serie de log-retornos (primera observación = 0)
    """
    ret = np.log(df[columna]) - np.log(df[columna].shift(1))
    ret.iloc[0] = 0
    return ret


In [ ]:
# ── Ejemplo: calcular retornos con Close ──
df_fred['ret'] = log_retornos(df_fred, 'Close')

# Con Adj Close (para acciones de Yahoo Finance):
# df_yf['ret'] = log_retornos(df_yf, 'Adj Close')


In [ ]:
# ── Log-retornos al cuadrado ──
# Proxy de la varianza instantánea.
# Si r² muestra autocorrelación → la volatilidad tiene memoria (efecto ARCH).

df_fred['ret2'] = df_fred['ret'] ** 2


In [ ]:
# ── Retorno anualizado ──
ret_anual = np.mean(df_fred['ret']) * 252
print(f"Retorno anualizado (log): {ret_anual:.4%}")


In [ ]:
# ── Volatilidad histórica ──
# Supone que σ es constante en toda la muestra.

def volatilidad_historica(ret):
    """
    Calcula la volatilidad diaria y la anualiza (×√252).
    
    Parámetros:
        ret : Serie de log-retornos
    Retorna:
        (vol_diaria, vol_anual)
    """
    vol_d = np.std(ret)
    vol_a = vol_d * np.sqrt(252)
    return vol_d, vol_a


In [ ]:
vol_d, vol_a = volatilidad_historica(df_fred['ret'])
print(f"Volatilidad diaria:      {vol_d:.4%}")
print(f"Volatilidad anualizada:  {vol_a:.4%}")


In [ ]:
# ── Gráfico de precios ──
plt.figure(figsize=(14, 4))
plt.plot(df_fred['Close'], color='darkblue', linewidth=0.8)
plt.title('Precio')
plt.ylabel('Precio')
plt.xlabel('Fecha')
ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()


In [ ]:
# ── Gráfico de retornos ──
plt.figure(figsize=(14, 4))
plt.plot(df_fred['ret'], color='red', linewidth=0.5)
plt.title('Log-Retornos')
plt.ylabel('Retorno')
plt.xlabel('Fecha')
ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()


In [ ]:
# ── Histograma de retornos ──
k = int(math.sqrt(len(df_fred['ret'])))

plt.figure(figsize=(10, 4))
plt.hist(df_fred['ret'], bins=k, edgecolor='black', linewidth=0.5)
plt.xlabel('Retorno')
plt.ylabel('Frecuencia')
plt.title('Histograma de Log-Retornos')
plt.tight_layout()
plt.show()


## 3. Pruebas estadísticas

Regla general: si **p < 0.05** → se rechaza H₀. Si **p ≥ 0.05** → no se rechaza H₀.

Nunca se dice "se acepta H₀", solo "no se rechaza".


### 3.1 Jarque-Bera - ¿los retornos son normales?

- **H₀**: los retornos son normales (asimetría = 0, exceso de curtosis = 0).
- **H₁**: los retornos no son normales.
- Si rechaza → colas pesadas y/o asimetría. BSM supone normalidad, así que es una simplificación.


In [ ]:
def prueba_jarque_bera(ret):
    """
    Prueba de normalidad Jarque-Bera.
    Imprime estadístico, p-valor, asimetría y curtosis.
    """
    jb_stat, jb_p = stats.jarque_bera(ret)
    skew = stats.skew(ret)           # asimetría
    kurt = stats.kurtosis(ret)       # exceso de curtosis
    
    print(f"Jarque-Bera = {jb_stat:.2f} | p-valor = {jb_p:.2e}")
    print(f"Asimetría (S) = {skew:.3f}")
    print(f"Exceso de curtosis (C) = {kurt:.2f}")
    
    if jb_p < 0.05:
        print("→ RECHAZA H₀: los retornos NO son normales.")
    else:
        print("→ NO se rechaza H₀: sin evidencia contra normalidad.")


In [ ]:
prueba_jarque_bera(df_fred['ret'])


### 3.2 Ljung-Box - ¿hay autocorrelación?

Se aplica dos veces:
- **Sobre $r_t$**: ¿el retorno de hoy depende de retornos pasados? (dependencia en media)
- **Sobre $r_t^2$**: ¿la varianza de hoy depende de varianzas pasadas? (efecto ARCH)

Si la segunda rechaza H₀ → la volatilidad no es constante → se justifica usar GARCH.


In [ ]:
def prueba_ljung_box(serie, lags=5, nombre="serie"):
    """
    Prueba de Ljung-Box para autocorrelación.
    
    Parámetros:
        serie  : Serie a evaluar (retornos o retornos²)
        lags   : número de rezagos
        nombre : etiqueta para el print
    """
    lb = sm.stats.diagnostic.acorr_ljungbox(serie, lags=lags)
    Q = lb.iloc[-1, 0]
    p = lb.iloc[-1, 1]
    
    print(f"Ljung-Box {nombre} ({lags} lags): Q = {Q:.2f} | p = {p:.4e}")
    
    if p < 0.05:
        print(f"→ RECHAZA H₀: hay autocorrelación en {nombre}.")
    else:
        print(f"→ NO se rechaza H₀: {nombre} es independiente.")
    
    return Q, p


In [ ]:
# Ljung-Box sobre retornos
prueba_ljung_box(df_fred['ret'], lags=5, nombre="r_t")


In [ ]:
# Ljung-Box sobre retornos² (prueba de efecto ARCH)
prueba_ljung_box(df_fred['ret2'], lags=5, nombre="r_t²")


### 3.3 Engle ARCH-LM - segunda opinión sobre efecto ARCH

- **H₀**: no hay efectos ARCH (todos los coeficientes de la regresión de $r^2$ sobre sus rezagos = 0).
- **H₁**: hay efectos ARCH.
- Confirma o contradice lo que dijo Ljung-Box sobre $r_t^2$.


In [ ]:
def prueba_engle_arch(ret, nlags=5):
    """
    Prueba de Engle ARCH-LM.
    
    Parámetros:
        ret   : Serie de retornos
        nlags : número de rezagos
    """
    resultado = het_arch(ret, nlags=nlags)
    lm_stat = resultado[0]
    lm_p = resultado[1]
    
    print(f"Engle ARCH-LM ({nlags} lags): LM = {lm_stat:.2f} | p = {lm_p:.2e}")
    
    if lm_p < 0.05:
        print("→ RECHAZA H₀: hay efecto ARCH.")
    else:
        print("→ NO se rechaza H₀: sin efecto ARCH.")


In [ ]:
prueba_engle_arch(df_fred['ret'], nlags=5)


### 3.4 Autocorrelogramas (ACF)

In [ ]:
# ACF de retornos y retornos² lado a lado
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(df_fred['ret'], lags=21, ax=axes[0], title='ACF de r_t')
plot_acf(df_fred['ret2'], lags=21, ax=axes[1], title='ACF de r_t²')
plt.tight_layout()
plt.show()


## 4. GARCH(1,1) - Volatilidad condicional

A diferencia de la volatilidad histórica (σ constante), el GARCH modela una volatilidad que cambia en el tiempo:

$$\sigma_t^2 = \omega + \alpha \cdot r_{t-1}^2 + \beta \cdot \sigma_{t-1}^2$$

- **ω**: constante
- **α**: peso del shock reciente ($r_{t-1}^2$)
- **β**: peso de la varianza pasada ($\sigma_{t-1}^2$)
- **Persistencia** (α + β): qué tan rápido se disipan los choques. Cerca de 1 = duran mucho.
- **σ largo plazo** = $\sqrt{\omega / (1 - \alpha - \beta)}$

Después de ajustar, se verifica con **Ljung-Box sobre los residuales² (5 lags)**:
- p ≥ 0.05 → el modelo capturó la heterocedasticidad (bueno).
- p < 0.05 → queda ARCH sin modelar (limitación).


In [ ]:
def ajustar_garch(ret):
    """
    Ajusta un GARCH(1,1) a los retornos.
    
    Parámetros:
        ret : Serie de log-retornos
    Retorna:
        res : objeto de resultados del modelo
    """
    model = arch_model(ret, mean='Zero', vol='GARCH', p=1, q=1, rescale=False)
    res = model.fit(disp='off')
    print(res.summary())
    return res


In [ ]:
res = ajustar_garch(df_fred['ret'])


In [ ]:
# ── Parámetros estimados ──
omega = res.params['omega']
alpha = res.params['alpha[1]']
beta  = res.params['beta[1]']
persist = alpha + beta

print(f"ω = {omega:.6e}")
print(f"α = {alpha:.4f}")
print(f"β = {beta:.4f}")
print(f"Persistencia (α + β) = {persist:.4f}")
print(f"σ largo plazo = {np.sqrt(omega / (1 - persist)) * np.sqrt(252):.4%}")


In [ ]:
# ── Ljung-Box sobre residuales estandarizados² ──
# Aquí NO rechazar es la buena noticia.
prueba_ljung_box(res.std_resid ** 2, lags=5, nombre="residuales²")


In [ ]:
# ── Último valor de volatilidad anual ──
vol_garch = res.conditional_volatility * np.sqrt(252)
sigma_final = vol_garch.iloc[-1]

print(f"Último valor de volatilidad anual (GARCH): σ = {sigma_final:.4%}")


In [ ]:
# ── Gráfico de volatilidad condicional ──
plt.figure(figsize=(14, 4))
plt.plot(vol_garch, color='darkred', linewidth=0.8)
plt.title('Volatilidad Anualizada - GARCH(1,1)')
plt.ylabel('σ anual')
plt.xlabel('Fecha')
ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()


## 5. Futuros y forwards - 6 escenarios

Un **futuro** (o forward) es un contrato para comprar/vender un activo a un precio fijo en una fecha futura.

La idea base es **cost-of-carry**: $F_0 = S_0 \cdot e^{rT}$, y se suma o resta según lo que pase con el activo mientras lo guardas.

**Regla**: lo que cuesta mantener el activo (almacenamiento) **suma**. Lo que rinde (dividendo, tasa foránea, convenience yield) **resta**.


In [ ]:
# ══════════════════════════════════════════════
# Escenario 1: Activo financiero, sin ingresos
# F₀ = S₀ · e^(rT)
# Ej: acción sin dividendos, oro financiero
# ══════════════════════════════════════════════

def futuro_financiero(S0, r, T):
    """
    Parámetros:
        S0 : precio spot
        r  : tasa libre de riesgo (c.c.)
        T  : tiempo a vencimiento (años)
    """
    return S0 * np.exp(r * T)


In [ ]:
# ══════════════════════════════════════════════
# Escenario 2: Activo con ingreso conocido (dividendo)
# F₀ = (S₀ − I) · e^(rT)
# I = valor presente de los dividendos
# Ej: acción que paga dividendos conocidos
# ══════════════════════════════════════════════

def futuro_dividendo(S0, r, T, I):
    """
    Parámetros:
        S0 : precio spot
        r  : tasa libre de riesgo (c.c.)
        T  : tiempo a vencimiento (años)
        I  : valor presente de los dividendos
    """
    return (S0 - I) * np.exp(r * T)


In [ ]:
# ══════════════════════════════════════════════
# Escenario 3: Commodity, almacenamiento como TASA (%)
# F₀ = S₀ · e^((r+u)T)
# u entra al exponente junto con r
# Ej: diésel del parcial, maíz
# ══════════════════════════════════════════════

def futuro_almacen_tasa(S0, r, u, T):
    """
    Parámetros:
        S0 : precio spot
        r  : tasa libre de riesgo (c.c.)
        u  : costo de almacenamiento (% anual)
        T  : tiempo a vencimiento (años)
    """
    return S0 * np.exp((r + u) * T)


In [ ]:
# ══════════════════════════════════════════════
# Escenario 4: Commodity, almacenamiento como MONTO ($)
# F₀ = (S₀ + U) · e^(rT)
# U se suma al spot, NO al exponente
# Ej: UniPollo (maíz)
# ══════════════════════════════════════════════

def futuro_almacen_monto(S0, r, T, U):
    """
    Parámetros:
        S0 : precio spot
        r  : tasa libre de riesgo (c.c.)
        T  : tiempo a vencimiento (años)
        U  : VP del costo de almacenamiento ($)
    """
    return (S0 + U) * np.exp(r * T)


In [ ]:
# ══════════════════════════════════════════════
# Escenario 5: Commodity con convenience yield
# F₀ = S₀ · e^((r+u−y)T)
# y = beneficio por tener el commodity físico
# ══════════════════════════════════════════════

def futuro_convenience(S0, r, u, y, T):
    """
    Parámetros:
        S0 : precio spot
        r  : tasa libre de riesgo (c.c.)
        u  : costo de almacenamiento (% anual)
        y  : convenience yield (% anual)
        T  : tiempo a vencimiento (años)
    """
    return S0 * np.exp((r + u - y) * T)


In [ ]:
# ══════════════════════════════════════════════
# Escenario 6: Divisas (FX)
# F₀ = S₀ · e^((r − r_f)T)
# La tasa foránea actúa como un dividendo continuo
# Ej: USD/COP
# ══════════════════════════════════════════════

def futuro_fx(S0, r, rf, T):
    """
    Parámetros:
        S0 : precio spot (ej: COP/USD)
        r  : tasa doméstica (c.c.)
        rf : tasa foránea (c.c.)
        T  : tiempo a vencimiento (años)
    """
    return S0 * np.exp((r - rf) * T)


## 6. Black-Scholes-Merton - 4 variantes

Una **opción** es un derecho (no obligación) de comprar o vender un activo a un precio fijo (strike K) en una fecha futura.

- **Call**: derecho a **comprar**. Payoff = max(S_T − K, 0).
- **Put**: derecho a **vender**. Payoff = max(K − S_T, 0).

BSM calcula el precio justo de una opción europea. La fórmula general es:

$$c = e^{-rT}[F_0 \cdot N(d_1) - K \cdot N(d_2)]$$

Lo único que cambia entre variantes es qué es $F_0$ (el forward implícito del activo).


### 6.1 Acción sin dividendos
$F_0 = S_0 \cdot e^{rT}$


In [ ]:
def bsm_accion(S, K, r, T, sigma, option):
    """
    BSM para acción sin dividendos.
    
    Parámetros:
        S      : precio spot
        K      : strike
        r      : tasa libre de riesgo (c.c.)
        T      : tiempo a vencimiento (años)
        sigma  : volatilidad anual
        option : 'call' o 'put'
    """
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    if option == 'call':
        return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    elif option == 'put':
        return K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)


### 6.2 Acción / índice con dividend yield (q)
$F_0 = S_0 \cdot e^{(r-q)T}$. El dividendo **resta** en el exponente.


In [ ]:
def bsm_indice(S, K, r, q, T, sigma, option):
    """
    BSM para acción/índice con dividend yield continuo.
    
    Parámetros:
        S      : precio spot
        K      : strike
        r      : tasa libre de riesgo (c.c.)
        q      : dividend yield continuo
        T      : tiempo a vencimiento (años)
        sigma  : volatilidad anual
        option : 'call' o 'put'
    """
    d1 = (np.log(S / K) + (r - q + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    if option == 'call':
        return np.exp(-r * T) * (S * np.exp((r - q) * T) * norm.cdf(d1) - K * norm.cdf(d2))
    elif option == 'put':
        return np.exp(-r * T) * (K * norm.cdf(-d2) - S * np.exp((r - q) * T) * norm.cdf(-d1))


### 6.3 Commodity con almacenamiento (u)
$F_0 = S_0 \cdot e^{(r+u)T}$. El almacenamiento **suma** en el exponente. Es la variante del parcial (diésel).


In [ ]:
def bsm_commodity(S, K, r, u, T, sigma, option):
    """
    BSM para commodity con costo de almacenamiento como tasa.
    
    Parámetros:
        S      : precio spot
        K      : strike
        r      : tasa libre de riesgo (c.c.)
        u      : costo de almacenamiento (% anual)
        T      : tiempo a vencimiento (años)
        sigma  : volatilidad anual
        option : 'call' o 'put'
    """
    d1 = (np.log(S / K) + (r + u + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    if option == 'call':
        return np.exp(-r * T) * (S * np.exp((r + u) * T) * norm.cdf(d1) - K * norm.cdf(d2))
    elif option == 'put':
        return np.exp(-r * T) * (K * norm.cdf(-d2) - S * np.exp((r + u) * T) * norm.cdf(-d1))


### 6.4 Divisa (FX)
$F_0 = S_0 \cdot e^{(r - r_f)T}$. La tasa foránea **resta** (como un dividendo).


In [ ]:
def bsm_fx(S, K, r, rf, T, sigma, option):
    """
    BSM para opciones sobre divisas.
    
    Parámetros:
        S      : precio spot (ej: COP/USD)
        K      : strike
        r      : tasa doméstica (c.c.)
        rf     : tasa foránea (c.c.)
        T      : tiempo a vencimiento (años)
        sigma  : volatilidad anual
        option : 'call' o 'put'
    """
    d1 = (np.log(S / K) + (r - rf + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    if option == 'call':
        return np.exp(-r * T) * (S * np.exp((r - rf) * T) * norm.cdf(d1) - K * norm.cdf(d2))
    elif option == 'put':
        return np.exp(-r * T) * (K * norm.cdf(-d2) - S * np.exp((r - rf) * T) * norm.cdf(-d1))


### 6.5 Paridad Put-Call

Sirve para verificar que los cálculos de call y put son consistentes.

| Subyacente | Paridad |
|---|---|
| Acción | c + K·e⁻ʳᵀ = p + S₀ |
| Índice (q) | c + K·e⁻ʳᵀ = p + S₀·e⁻ᑫᵀ |
| Futuro | c + K·e⁻ʳᵀ = p + F₀·e⁻ʳᵀ |
| Divisa | c + K·e⁻ʳᵀ = p + S₀·e⁻ʳᶠᵀ |

Si ambos lados no coinciden, hay un error en el cálculo.


## 7. Árboles binomiales

El árbol modela el precio subiendo (×u) o bajando (×d) en cada paso:

- $u = e^{\sigma\sqrt{\Delta t}}$, $d = 1/u$
- $p = \frac{a - d}{u - d}$ (probabilidad neutral al riesgo)
- **Descuento** siempre con $e^{-r\Delta t}$ (nunca con $a$)

Lo que cambia es el **factor de crecimiento $a$**:

| Subyacente | Factor a |
|---|---|
| Acción sin dividendos | $e^{r\Delta t}$ |
| Acción con dividendos | $e^{(r-q)\Delta t}$ |
| Divisa | $e^{(r-r_f)\Delta t}$ |
| Commodity con almacenamiento | $e^{(r+u)\Delta t}$ |

Para **opciones americanas**, en cada nodo se compara el valor de ejercer ahora vs. esperar.


In [ ]:
def binomial_europeo(S, K, r, T, sigma, N, option, ajuste=0):
    """
    Árbol binomial para opción EUROPEA.
    
    Parámetros:
        S      : precio spot
        K      : strike
        r      : tasa libre de riesgo (c.c.)
        T      : tiempo a vencimiento (años)
        sigma  : volatilidad anual
        N      : número de pasos
        option : 'call' o 'put'
        ajuste : término que modifica el factor a:
                 0 para acción sin dividendos (a = e^(r·dt))
                 -q para índice con dividendos (a = e^((r-q)·dt))
                 -rf para divisa (a = e^((r-rf)·dt))
                 +u para commodity (a = e^((r+u)·dt))
    """
    dt = T / N
    up = np.exp(sigma * np.sqrt(dt))
    dn = 1 / up
    a  = np.exp((r + ajuste) * dt)    # factor de crecimiento
    p  = (a - dn) / (up - dn)
    disc = np.exp(-r * dt)            # descuento siempre con r

    # Precios en nodos finales
    ST = np.array([S * up**j * dn**(N - j) for j in range(N + 1)])

    # Payoff final
    if option == 'call':
        V = np.maximum(ST - K, 0)
    else:
        V = np.maximum(K - ST, 0)

    # Backward induction (sin ejercicio temprano)
    for i in range(N - 1, -1, -1):
        V = disc * (p * V[1:i+2] + (1 - p) * V[0:i+1])

    return V[0]


In [ ]:
def binomial_americano(S, K, r, T, sigma, N, option, ajuste=0):
    """
    Árbol binomial para opción AMERICANA (ejercicio temprano).
    
    Mismos parámetros que binomial_europeo.
    En cada nodo: V = max(valor intrínseco, valor de esperar).
    """
    dt = T / N
    up = np.exp(sigma * np.sqrt(dt))
    dn = 1 / up
    a  = np.exp((r + ajuste) * dt)
    p  = (a - dn) / (up - dn)
    disc = np.exp(-r * dt)

    # Precios en nodos finales
    ST = np.array([S * up**j * dn**(N - j) for j in range(N + 1)])

    # Payoff final
    if option == 'call':
        V = np.maximum(ST - K, 0)
    else:
        V = np.maximum(K - ST, 0)

    # Backward induction CON ejercicio temprano
    for i in range(N - 1, -1, -1):
        Si = np.array([S * up**j * dn**(i - j) for j in range(i + 1)])
        V_hold = disc * (p * V[1:i+2] + (1 - p) * V[0:i+1])
        
        if option == 'call':
            V_exercise = np.maximum(Si - K, 0)
        else:
            V_exercise = np.maximum(K - Si, 0)
        
        V = np.maximum(V_hold, V_exercise)

    return V[0]


### Cómo usar el parámetro `ajuste`

```python
# Acción sin dividendos (ajuste = 0, por defecto):
binomial_europeo(S, K, r, T, sigma, N, 'call')

# Índice con dividendos (ajuste = -q):
binomial_europeo(S, K, r, T, sigma, N, 'call', ajuste=-q)

# Divisa (ajuste = -rf):
binomial_europeo(S, K, r, T, sigma, N, 'call', ajuste=-rf)

# Commodity con almacenamiento (ajuste = +u):
binomial_americano(S, K, r, T, sigma, N, 'put', ajuste=u)
```


## 8. Estrategias con opciones

Una **estrategia** combina calls y/o puts para lograr un perfil de riesgo específico.

| Estrategia | Composición | Vista |
|---|---|---|
| **Bull spread** (calls) | Long call K₁ + Short call K₂ (K₁ < K₂) | Alcista moderado |
| **Bear spread** (puts) | Long put K₂ + Short put K₁ (K₁ < K₂) | Bajista moderado |
| **Butterfly** | Long K₁ + Long K₃ + Short 2×K₂ | Apuesta a que S_T ≈ K₂ |
| **Straddle** | Long call K + Long put K (mismo K) | Apuesta a movimiento grande |
| **Strangle** | Long put K₁ + Long call K₂ (K₁ < K₂) | Como straddle, más barato |
| **Collar** | Long call K₂ + Short put K₁ (K₁ < K₂) | Acota costo entre K₁ y K₂ |


### Collar (la estrategia del parcial)

- Se **compra un call** (K₂ = 14,000): protege contra subidas.
- Se **vende un put** (K₁ = 11,000): financia parte del call, pero te expone a caídas debajo de K₁.
- **Prima neta** = precio call − precio put.
- Si la prima neta es positiva → pagas por la estrategia (protección cuesta más que el ingreso del put).
- Si es negativa → recibes dinero neto (el put vendido vale más que el call comprado).


In [ ]:
def prima_collar(S, K_put, K_call, r, T, sigma, bsm_func, **kwargs):
    """
    Calcula la prima neta de un collar.
    
    Parámetros:
        S       : precio spot
        K_put   : strike del put (short)
        K_call  : strike del call (long)
        r       : tasa libre de riesgo
        T       : tiempo a vencimiento
        sigma   : volatilidad
        bsm_func: función BSM a usar (bsm_commodity, bsm_fx, etc.)
        **kwargs: parámetros adicionales (u, rf, q, según la variante)
    """
    call = bsm_func(S, K_call, r, T=T, sigma=sigma, option='call', **kwargs)
    put  = bsm_func(S, K_put,  r, T=T, sigma=sigma, option='put',  **kwargs)
    
    print(f"Call (K={K_call:,}): ${call:,.2f}")
    print(f"Put  (K={K_put:,}):  ${put:,.2f}")
    print(f"Prima neta collar:   ${call - put:,.2f}")
    
    return call - put


---
## Resumen de funciones

| Función | Uso |
|---|---|
| `log_retornos(df, col)` | Calcula log-retornos |
| `volatilidad_historica(ret)` | σ constante (diaria y anual) |
| `prueba_jarque_bera(ret)` | ¿Normalidad? |
| `prueba_ljung_box(serie, lags, nombre)` | ¿Autocorrelación? |
| `prueba_engle_arch(ret, nlags)` | ¿Efecto ARCH? |
| `ajustar_garch(ret)` | GARCH(1,1) |
| `futuro_financiero / _dividendo / _almacen_tasa / _almacen_monto / _convenience / _fx` | 6 escenarios de futuros |
| `bsm_accion / _indice / _commodity / _fx` | 4 variantes de BSM |
| `binomial_europeo(S, K, r, T, σ, N, opt, ajuste)` | Árbol binomial europeo |
| `binomial_americano(S, K, r, T, σ, N, opt, ajuste)` | Árbol binomial americano |
| `prima_collar(S, K_put, K_call, ...)` | Prima neta del collar |
